# 🏆 Notebook 05 — Client Health Score & Executive Dashboard
## Client Delivery Intelligence — IT Services Analytics

**Business Question:** *Can we score every client account on a single 0–100 health metric — combining SLA, delivery, utilisation, and revenue signals — so account managers know exactly where to focus?*

**This is the 'Watermelon Account' detector** — accounts that look green on status reports but are actually red underneath.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

BLUE='#1A3C6E'; RED='#C0392B'; GREEN='#27AE60'; ORANGE='#E67E22'; GRAY='#95A5A6'; LIGHT='#EEF4FB'
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})

clients    = pd.read_csv('../data/clients.csv')
projects   = pd.read_csv('../data/projects.csv')
tickets    = pd.read_csv('../data/tickets.csv')
timesheets = pd.read_csv('../data/timesheets.csv')

print('Data loaded. Building Client Health Scores...')

In [ ]:
# ── Build Client Health Score ──────────────────────────────────────────────
# Score = weighted composite of 4 dimensions (0-100 each)

# 1. SLA Health (30% weight)
sla_health = tickets.groupby('client_id').agg(
    breach_rate=('sla_breached','mean'),
    avg_csat=('csat_score','mean'),
    fcr_rate=('first_contact_resolved','mean'),
    escalation_rate=('escalated','mean')
).reset_index()
sla_health['sla_score'] = (
    (1 - sla_health['breach_rate']) * 40 +
    (sla_health['avg_csat'] / 5) * 30 +
    sla_health['fcr_rate'] * 20 +
    (1 - sla_health['escalation_rate']) * 10
).clip(0, 100)

# 2. Delivery Health (35% weight)
delivery_health = projects.groupby('client_id').agg(
    on_track_rate=('status', lambda x: (x=='On Track').mean()),
    avg_overrun=('cost_overrun_pct','mean'),
    avg_delay=('delay_days','mean'),
    milestone_hit=('milestone_hit_rate_pct','mean')
).reset_index()
delivery_health['delivery_score'] = (
    delivery_health['on_track_rate'] * 35 +
    (1 - (delivery_health['avg_overrun'] / 100).clip(0,1)) * 30 +
    (1 - (delivery_health['avg_delay'] / 60).clip(0,1)) * 15 +
    (delivery_health['milestone_hit'] / 100) * 20
).clip(0, 100)

# 3. Revenue Health (20% weight)
rev_health = timesheets.groupby('client_id').agg(
    total_billed=('billable_hrs','sum'),
    total_unbilled=('unbilled_hrs','sum'),
    avg_util=('utilisation_pct','mean')
).reset_index()
rev_health['leakage_rate'] = rev_health['total_unbilled'] / (rev_health['total_billed'] + rev_health['total_unbilled'] + 1)
rev_health['revenue_score'] = (
    (1 - rev_health['leakage_rate']) * 60 +
    ((rev_health['avg_util'] / 100).clip(0,1)) * 40
).clip(0, 100)

# 4. Ticket Volume Trend (15% weight) — is ticket backlog growing?
recent_tickets = tickets[tickets['week'] >= 40].groupby('client_id')['ticket_id'].count().reset_index()
recent_tickets.columns = ['client_id','recent_volume']
early_tickets = tickets[tickets['week'] <= 12].groupby('client_id')['ticket_id'].count().reset_index()
early_tickets.columns = ['client_id','early_volume']
trend = recent_tickets.merge(early_tickets, on='client_id', how='left')
trend['volume_growth'] = (trend['recent_volume'] - trend['early_volume']) / (trend['early_volume'] + 1)
trend['trend_score'] = (1 - trend['volume_growth'].clip(0,1)) * 100

# ── Merge all dimensions ──────────────────────────────────────────────────
health = clients[['client_id','client_name','industry','tier','annual_revenue_lakh']].copy()
health = health.merge(sla_health[['client_id','sla_score','breach_rate','avg_csat']], on='client_id', how='left')
health = health.merge(delivery_health[['client_id','delivery_score','on_track_rate','avg_overrun']], on='client_id', how='left')
health = health.merge(rev_health[['client_id','revenue_score','leakage_rate']], on='client_id', how='left')
health = health.merge(trend[['client_id','trend_score','volume_growth']], on='client_id', how='left')
health = health.fillna(health.median(numeric_only=True))

# Composite Health Score
health['health_score'] = (
    health['sla_score']      * 0.30 +
    health['delivery_score'] * 0.35 +
    health['revenue_score']  * 0.20 +
    health['trend_score']    * 0.15
).round(1)

health['health_band'] = pd.cut(health['health_score'],
    bins=[0,40,60,75,100], labels=['Critical','At Risk','Healthy','Excellent'])

# Watermelon flag: on_track_rate > 60% BUT health_score < 55
health['watermelon_flag'] = ((health['on_track_rate'] > 0.6) & (health['health_score'] < 55)).astype(int)

print('Health scores computed.')
print(health['health_band'].value_counts())
print(f"\nWatermelon accounts detected: {health['watermelon_flag'].sum()}")

In [ ]:
# ── Figure 5: Full Executive Health Dashboard ──────────────────────────────
fig = plt.figure(figsize=(20,14), facecolor='white')
fig.text(0.5,0.97,'CLIENT HEALTH SCORE — ACCOUNT INTELLIGENCE DASHBOARD',
         ha='center',fontsize=20,fontweight='bold',color=BLUE)
fig.text(0.5,0.945,'Composite Score: SLA (30%) + Delivery (35%) + Revenue (20%) + Trend (15%)',
         ha='center',fontsize=11,color=GRAY)
gs = gridspec.GridSpec(3,4,figure=fig,hspace=0.55,wspace=0.4,top=0.93,bottom=0.05)

# KPI cards
band_counts = health['health_band'].value_counts()
kpis = [
    (str(band_counts.get('Excellent',0)+band_counts.get('Healthy',0)),'Healthy Accounts',GREEN,'Score ≥ 75'),
    (str(band_counts.get('At Risk',0)),'At Risk Accounts',ORANGE,'Score 60–75'),
    (str(band_counts.get('Critical',0)),'Critical Accounts',RED,'Score < 60'),
    (str(health['watermelon_flag'].sum()),'Watermelon Accounts',RED,'Look green, are red'),
]
for i,(val,label,color,sub) in enumerate(kpis):
    ax = fig.add_subplot(gs[0,i])
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    ax.add_patch(FancyBboxPatch((0.05,0.05),0.9,0.9,boxstyle='round,pad=0.05',facecolor=LIGHT,edgecolor=color,linewidth=2))
    ax.text(0.5,0.68,val,ha='center',va='center',fontsize=22,fontweight='bold',color=color)
    ax.text(0.5,0.42,label,ha='center',va='center',fontsize=10,color=BLUE,fontweight='bold')
    ax.text(0.5,0.18,sub,ha='center',va='center',fontsize=8.5,color=GRAY,style='italic')

# Health score distribution
ax2 = fig.add_subplot(gs[1,0:2])
health_sorted = health.sort_values('health_score')
bar_colors = [RED if s<60 else ORANGE if s<75 else GREEN for s in health_sorted['health_score']]
ax2.bar(range(len(health_sorted)), health_sorted['health_score'], color=bar_colors, alpha=0.85)
ax2.axhline(75, color=GREEN, linestyle='--', linewidth=1.5, label='Healthy threshold (75)')
ax2.axhline(60, color=RED, linestyle='--', linewidth=1.5, label='At Risk threshold (60)')
# Mark watermelons
wm_idx = [i for i,(wm,s) in enumerate(zip(health_sorted['watermelon_flag'], health_sorted['health_score'])) if wm==1]
ax2.scatter(wm_idx, [health_sorted['health_score'].iloc[i]+2 for i in wm_idx],
            marker='*', color=ORANGE, s=100, zorder=5, label='Watermelon account')
ax2.set_xlabel('Client Accounts (sorted by health score)')
ax2.set_ylabel('Health Score (0–100)')
ax2.set_title('Client Health Score — All 50 Accounts\n★ = Watermelon Account (looks healthy, is at risk)',fontweight='bold',color=BLUE)
ax2.legend(fontsize=8)
ax2.set_facecolor('#FAFCFF')

# Score components radar-style bar
ax3 = fig.add_subplot(gs[1,2])
bottom10 = health.nsmallest(10,'health_score')
dims = ['sla_score','delivery_score','revenue_score','trend_score']
dim_labels = ['SLA','Delivery','Revenue','Trend']
x = np.arange(len(dim_labels))
avg_all = [health[d].mean() for d in dims]
avg_bad = [bottom10[d].mean() for d in dims]
w=0.35
ax3.bar(x-w/2, avg_all, w, color=BLUE, alpha=0.85, label='All accounts avg')
ax3.bar(x+w/2, avg_bad, w, color=RED, alpha=0.85, label='Bottom 10 accounts')
ax3.set_xticks(x); ax3.set_xticklabels(dim_labels)
ax3.set_ylabel('Score (0–100)')
ax3.set_title('Score Breakdown\nAll vs At-Risk Accounts',fontweight='bold',color=BLUE)
ax3.legend(fontsize=8)

# Industry health heatmap
ax4 = fig.add_subplot(gs[1,3])
ind_health = health.groupby('industry')['health_score'].mean().sort_values()
colors_ind = [RED if v<60 else ORANGE if v<75 else GREEN for v in ind_health]
ax4.barh(ind_health.index,ind_health.values,color=colors_ind,alpha=0.85)
for i,v in enumerate(ind_health.values):
    ax4.text(v+0.5,i,f'{v:.0f}',va='center',fontsize=9,fontweight='bold')
ax4.axvline(75,color=GREEN,linestyle='--',linewidth=1.2,alpha=0.7)
ax4.set_title('Avg Health Score\nby Industry',fontweight='bold',color=BLUE)
ax4.set_xlim(0,105)

# Bottom 10 accounts table
ax5 = fig.add_subplot(gs[2,:])
ax5.axis('off')
bottom10_display = health.nsmallest(10,'health_score')[[
    'client_name','industry','tier','health_score','breach_rate',
    'avg_overrun','leakage_rate','watermelon_flag','health_band'
]].copy()
bottom10_display.columns = ['Account','Industry','Tier','Health Score','SLA Breach %','Avg Overrun %','Revenue Leakage %','Watermelon?','Status']
bottom10_display['SLA Breach %'] = (bottom10_display['SLA Breach %']*100).round(1)
bottom10_display['Revenue Leakage %'] = (bottom10_display['Revenue Leakage %']*100).round(1)
bottom10_display['Watermelon?'] = bottom10_display['Watermelon?'].map({1:'⚠ YES',0:'No'})

tbl = ax5.table(cellText=bottom10_display.values, colLabels=bottom10_display.columns,
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1,1.6)
for (r,c),cell in tbl.get_celld().items():
    if r==0:
        cell.set_facecolor(BLUE); cell.set_text_props(color='white',fontweight='bold')
    elif bottom10_display.iloc[r-1]['Status']=='Critical' if r>0 else False:
        cell.set_facecolor('#FFEBEB')
    elif r%2==0:
        cell.set_facecolor('#F5F8FF')
ax5.set_title('Bottom 10 Accounts — Immediate Action Required',fontweight='bold',color=RED,pad=15,fontsize=12)

plt.savefig('../docs/fig05_client_health_dashboard.png',dpi=150,bbox_inches='tight')
plt.show()
health.to_csv('../data/client_health_scores.csv',index=False)
print('Figure 05 saved. Health scores saved to data/client_health_scores.csv')

In [ ]:
# ── Final Business Summary ──────────────────────────────────────────────────
print('='*60)
print('CLIENT DELIVERY INTELLIGENCE — FINAL BUSINESS SUMMARY')
print('='*60)
print(f'\nPortfolio size      : {len(clients)} accounts, {len(projects)} projects')
print(f'Analysis period     : 52 weeks | {len(tickets):,} service tickets')
print(f'\nHEALTH SCORECARD:')
for band in ['Excellent','Healthy','At Risk','Critical']:
    count = (health['health_band']==band).sum()
    print(f'  {band:<12}: {count} accounts')
print(f'\nWatermelon accounts : {health["watermelon_flag"].sum()} (look green, are secretly red)')
print(f'\nTOP RISKS:')
print(f'  SLA breach rate    : {tickets.sla_breached.mean()*100:.1f}% (target: <5%)')
print(f'  Avg cost overrun   : {projects.cost_overrun_pct.mean():.1f}%')
print(f'  Revenue leakage    : {timesheets.unbilled_hrs.sum():,.0f} unbilled hours/year')
est_leakage = timesheets.unbilled_hrs.sum() * 2.5 / 100  # crores
print(f'  Estimated leakage  : ₹{est_leakage:.1f}Cr/year')
print(f'\nRECOMMENDATIONS:')
print(f'  1. Immediate escalation review for {(health["health_band"]=="Critical").sum()} critical accounts')
print(f'  2. Investigate {health["watermelon_flag"].sum()} watermelon accounts before client escalation')
print(f'  3. Deploy SLA early warning for P1/P2 tickets — 14-day backlog trend is top predictor')
print(f'  4. Address revenue leakage — ₹{est_leakage:.1f}Cr recoverable annually with timesheet discipline')